# Exploring the ATT&CK Knowledge Graph — Neo4j

This notebook **only reads** (`MATCH` queries, no `CREATE`/`MERGE`/`DELETE`), so unlike `load_neo4j.py` its cells are safe to re-run or run out of order; they cannot damage the data.

**Prerequisite**: `load_neo4j.py` has run successfully (the graph is loaded into your Neo4j). If not, run that first from a terminal — see `README.md`.

Run the cells from top to bottom. Kernel: choose the same Python venv used for `pip install -r requirements.txt` (`.venv` in this folder). Set `NEO4J_PASSWORD` in the environment, or enter the password at the prompt.

In [ ]:
import os
import getpass

import pandas as pd
from neo4j import GraphDatabase

URI = os.environ.get("NEO4J_URI", "bolt://localhost:7687")
USER = os.environ.get("NEO4J_USER", "neo4j")
PASSWORD = os.environ.get("NEO4J_PASSWORD") or getpass.getpass(
    "Neo4j password: "
)

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
driver.verify_connectivity()
print(f"Connected to {URI} as {USER}")


def run_df(query: str, **params) -> pd.DataFrame:
    """Run a Cypher query and return the result as a pandas DataFrame."""
    with driver.session() as session:
        result = session.run(query, **params)
        return pd.DataFrame([dict(record) for record in result])

## 1. Node & relationship counts

Compare the `count` column below with the **"Verification numbers"** table in `README.md`. If they differ, possible causes: (a) `load_neo4j.py` has not finished or partly failed, or (b) a different ATT&CK version was loaded (the paper uses v19.2, downloaded on 23 August 2026) — not automatically a bug.

In [ ]:
COUNT_QUERIES = {
    "Tactic": "MATCH (n:Tactic) RETURN count(n) AS c",
    "Technique (all)": "MATCH (n:Technique) RETURN count(n) AS c",
    "Parent technique": "MATCH (n:Technique {is_subtechnique: false}) RETURN count(n) AS c",
    "Sub-technique": "MATCH (n:Technique {is_subtechnique: true}) RETURN count(n) AS c",
    "DetectionStrategy": "MATCH (n:DetectionStrategy) RETURN count(n) AS c",
    "Analytic": "MATCH (n:Analytic) RETURN count(n) AS c",
    "DataComponent": "MATCH (n:DataComponent) RETURN count(n) AS c",
    "Rel. PART_OF": "MATCH ()-[r:PART_OF]->() RETURN count(r) AS c",
    "Rel. DETECTS": "MATCH ()-[r:DETECTS]->() RETURN count(r) AS c",
    "Rel. USES_LOG_SOURCE": "MATCH ()-[r:USES_LOG_SOURCE]->() RETURN count(r) AS c",
}

rows = []
for label, q in COUNT_QUERIES.items():
    with driver.session() as session:
        c = session.run(q).single()["c"]
    rows.append({"entity": label, "count": c})

df_counts = pd.DataFrame(rows)
df_counts

## 2. Distribution of techniques per tactic

Taxonomy sanity check — the 15 tactics of v19.2 include **"Stealth"** and **"Defense Impairment"** (the split of the old "Defense Evasion").

In [ ]:
df_tactic = run_df(
    """
    MATCH (te:Technique)-[:PART_OF]->(ta:Tactic)
    RETURN ta.name AS tactic, count(te) AS n_techniques
    ORDER BY n_techniques DESC
    """
)
df_tactic

## 3. Techniques whose detection relies on network traffic

The most relevant example for Maltrail (a traffic-based detector) — techniques with a `DetectionStrategy -> Analytic -> DataComponent` path to a "network traffic" data component.

**Definition of "network traffic" used here** (5 of the 6 active v19.2 data components whose name contains Network/DNS): `Network Connection Creation`, `Network Traffic Content`, `Network Traffic Flow`, `Active DNS`, `Passive DNS`. **`Network Share Access` is deliberately NOT included** here — it is more commonly instrumented through host logs (Windows Security/Sysmon) than through pure packet capture. If the team wants a looser definition (including `Network Share Access`), uncomment that line in the cell below — see also section 5 for a direct comparison of the two definitions.

In [ ]:
NETWORK_DATA_COMPONENTS = [
    'Network Connection Creation', 'Network Traffic Content',
    'Network Traffic Flow', 'Active DNS', 'Passive DNS',
    # 'Network Share Access',  # uncomment for a looser definition of "network traffic"
]

df_network_tech = run_df(
    """
    MATCH (te:Technique)<-[:DETECTS]-(:DetectionStrategy)-[:HAS_ANALYTIC]->(an:Analytic)
          -[:USES_LOG_SOURCE]->(dc:DataComponent)
    WHERE dc.name IN $names
    RETURN DISTINCT te.id AS technique_id, te.name AS technique_name
    ORDER BY te.id
    """,
    names=NETWORK_DATA_COMPONENTS,
)
print(f"{len(df_network_tech)} unique techniques detected through network-traffic data")
df_network_tech

## 4. Data components used most often by analytics

Candidate "bridges" between Maltrail alerts and the `DataComponent` nodes in this graph — see which are used most (top 20).

In [ ]:
df_dc = run_df(
    """
    MATCH (:Analytic)-[:USES_LOG_SOURCE]->(dc:DataComponent)
    RETURN dc.name AS data_component, count(*) AS n_analytics_using
    ORDER BY n_analytics_using DESC
    """
)
df_dc.head(20)

## 5. Cross-checking the 30.9% figure (Differentiator #6) directly from the database

**Update after investigation (23 August 2026):** the first version of this cell had a bug — its filter only captured 4 of the 6 active v19.2 data components whose name contains Network/DNS, so it gave 534/30.4% when it should have given 544/30.9% (the figure already used in the README and discussed as candidate Differentiator #6). The bug was in the notebook code, not in the data or in Neo4j.

The cell below now computes **both definitions explicitly** so that the difference is clearly visible, rather than a single number that looks final:
- **6 components** (including `Network Share Access`) → should be 544/1758 = 30.9% (the figure already used in the README)
- **5 components** (without `Network Share Access`, stricter "pure traffic") → should be 534/1758 = 30.4%

`Network Share Access` alone accounts for the whole difference of 10 analytics — `Passive DNS` turns out to add no new analytics at all (every analytic mentioning it is already counted through another component). This is a definitional decision, not a technical one — to be discussed with Dedy & Prof. Ashari before it goes into the paper.

In [ ]:
total_analytics = run_df("MATCH (n:Analytic) RETURN count(n) AS c").iloc[0]["c"]

BROAD_6 = [
    'Network Connection Creation', 'Network Traffic Content', 'Network Traffic Flow',
    'Active DNS', 'Passive DNS', 'Network Share Access',
]
NARROW_5 = [
    'Network Connection Creation', 'Network Traffic Content', 'Network Traffic Flow',
    'Active DNS', 'Passive DNS',
]

def count_network_analytics(names):
    return run_df(
        """
        MATCH (an:Analytic)-[:USES_LOG_SOURCE]->(dc:DataComponent)
        WHERE dc.name IN $names
        RETURN count(DISTINCT an) AS c
        """,
        names=names,
    ).iloc[0]["c"]

broad_count = count_network_analytics(BROAD_6)
narrow_count = count_network_analytics(NARROW_5)

print(f"Analytics in total                                      : {total_analytics}")
print(f"Analytics (6 components, incl. Network Share Access)    : {broad_count}  ({100*broad_count/total_analytics:.1f}%)")
print(f"Analytics (5 components, without Network Share Access)  : {narrow_count}  ({100*narrow_count/total_analytics:.1f}%)")
print()
print("Compare with README.md: 544/1758 = 30.9% (6-component definition).")

if broad_count != 544 or total_analytics != 1758:
    print()
    print("WARNING: the 6-component figure from the database differs from the README.")
    print("Possible causes: a different ATT&CK version was loaded than the one")
    print("downloaded on 23 August 2026 (v19.2), or a new Network/DNS-themed data")
    print("component exists that is not in the BROAD_6 list above -- if so, re-check")
    print("MATCH (dc:DataComponent) WHERE dc.name CONTAINS 'Network' OR")
    print("dc.name CONTAINS 'DNS' RETURN dc.name.")

## 6. Sub-techniques of a given technique

Change `TECHNIQUE_ID` to explore other techniques freely.

In [ ]:
TECHNIQUE_ID = "T1055"  # example: Process Injection

df_sub = run_df(
    """
    MATCH (sub:Technique)-[:SUBTECHNIQUE_OF]->(parent:Technique {id: $tid})
    RETURN sub.id AS sub_id, sub.name AS sub_name
    ORDER BY sub.id
    """
    ,
    tid=TECHNIQUE_ID,
)
df_sub

## 7. Example of a full relationship chain

`DataComponent -> Analytic -> DetectionStrategy -> Technique -> Tactic` — the relational pattern behind the "relational KG" claim (Differentiator #2), as opposed to a flat threshold table such as Kalutharage et al.

In [ ]:
df_chain = run_df(
    """
    MATCH (dc:DataComponent {name: 'Network Traffic Content'})
          <-[:USES_LOG_SOURCE]-(an:Analytic)
          <-[:HAS_ANALYTIC]-(ds:DetectionStrategy)
          -[:DETECTS]->(te:Technique)
          -[:PART_OF]->(ta:Tactic)
    RETURN dc.name AS data_component, an.id AS analytic_id, an.name AS analytic_name,
           ds.id AS detection_strategy_id, te.id AS technique_id,
           te.name AS technique_name, ta.name AS tactic
    LIMIT 10
    """
)
df_chain

## Closing notes

- All queries above are read-only — safe to re-run any number of times, unlike `load_neo4j.py`, where the order of cells/scripts matters.
- Take the results of section 1 (number verification) and section 5 (the 30.9% cross-check) to the discussion with Dedy & Prof. Ashari before candidate Differentiator #6 is added to the framework/paper.
- The `Analytic` descriptions (column `analytic_name` / `description`) are still **[NEEDS VERIFICATION]** regarding how much of them was authored with AI assistance by the MITRE team — do not quote them as "expert knowledge" in the draft without that qualification.

In [ ]:
driver.close()
print("Connection closed.")